# Week 16: Transfer learning, GPUs, multiple seeds, and calibration (milestone M6)

| Session | Topic | Time |
|---|---|---|
| Day 1 | Transfer learning: a pretrained ResNet-18, frozen vs fine-tuned | 2.5 h |
| Day 2 | GPU workflow: device-agnostic code, Colab/Kaggle, checkpoints | 2.5 h |
| Day 3 | Variance across seeds: never trust a single run | 1.5 h |
| Day 4 | Calibration: are the probabilities honest? ECE and reliability diagrams | 2 h |
| Buffer | Catch up on Weeks 13–16 | 1.5 h |

**Read:** CS231n notes, *Transfer Learning*; Guo et al. (2017), *On Calibration of Modern Neural Networks* (sections 1–3).

### Milestone M6
- [ ] `ResNetClassifier` in `medlab/imaging/models.py`, with `freeze_backbone` and `pretrained` options.
- [ ] Your code runs on CPU *and* GPU, chosen by `get_device()` alone.
- [ ] A results table: SmallCNN, ResNet (frozen) and ResNet (fine-tuned, on GPU) as validation AUROC **mean ± std over 3 seeds**.
- [ ] `medlab/imaging/calibration.py` with ECE, plus a reliability diagram of your best model.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

In [ ]:
import numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from course.data import load_medmnist, fake_mode
idata, imodels, itrain = fresh("medlab.imaging.data"), fresh("medlab.imaging.models"), fresh("medlab.imaging.train")
data = load_medmnist("pneumoniamnist", size=28)
ds = idata.make_tensors(data)
val_X, val_y = ds["val"].tensors[0], data["val"][1]
PRETRAINED = not fake_mode()        # the offline test mode can't download ImageNet weights
EPOCHS = 2 if fake_mode() else 5

---
# Day 1: Transfer learning (2.5 h)

A ResNet-18 trained on **ImageNet** (1.2M natural photos) has learnt general visual features: edges, textures, shapes. **Transfer learning** reuses them:
- **Feature extraction (frozen backbone):** train only the new final layer. It's fast and works with little data, and it's feasible on a CPU.
- **Fine-tuning:** also update the backbone, with a small learning rate. It's usually better when the domain differs (X-rays aren't photos!), but it needs a GPU and risks overfitting.

**Adapting the inputs:** ResNet expects 3-channel images of about 224×224. We have 1 channel at 28×28, so we **repeat** the channel 3 times and **upsample** to `input_size` (64 is a CPU-friendly compromise). In Week 19 you can try MedMNIST's native 224×224 version on a GPU.

### Exercise 16.1: `ResNetClassifier` in `capstone/medlab/imaging/models.py`
```python
class ResNetClassifier(nn.Module):
    def __init__(self, n_classes=2, pretrained=True, freeze_backbone=False, input_size=64)
    def forward(self, x)     # x: (N, 1, 28, 28) → upsample (F.interpolate, bilinear) → repeat to 3 channels → resnet
def count_trainable(model) -> int
```
Use `torchvision.models.resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)`, and replace `.fc` with `nn.Linear(512, n_classes)`. To freeze, set `requires_grad = False` on every backbone parameter, **then** create the new `fc` (new layers are trainable by default).

The first time `pretrained=True` runs, about 45 MB of weights download from `download.pytorch.org`.

In [ ]:
# Implement ResNetClassifier and count_trainable, then run the checker.

In [ ]:
imodels = fresh("medlab.imaging.models")
frozen = imodels.ResNetClassifier(pretrained=PRETRAINED, freeze_backbone=True)
full = imodels.ResNetClassifier(pretrained=PRETRAINED, freeze_backbone=False)
print(f"trainable: frozen {imodels.count_trainable(frozen):,}  vs  full {imodels.count_trainable(full):,}")
assert imodels.count_trainable(frozen) == 512 * 2 + 2
assert imodels.count_trainable(full) > 11_000_000
assert frozen(torch.zeros(4, 1, 28, 28)).shape == (4, 2)
print("✅ 16.1 passed")

---
# Day 2: The GPU workflow (2.5 h)

**Your laptop has no CUDA GPU**, which is fine. Write **device-agnostic** code once, and run it anywhere:
```python
device = get_device()            # "cuda" on Colab/Kaggle, "cpu" on your laptop
model.to(device); x = x.to(device)
```

### Exercise 16.2: `get_device()` in `medlab/imaging/train.py`
Return `"cuda"` if `torch.cuda.is_available()`, otherwise `"mps"` if Apple's backend is available, otherwise `"cpu"`. Make sure `fit` and `predict_proba` pass `device` through (they already take a `device` argument if you followed Week 15).

In [ ]:
# Add get_device to train.py, then run the checker.

In [ ]:
itrain = fresh("medlab.imaging.train")
dev = itrain.get_device()
assert dev in ("cuda", "mps", "cpu")
print("device:", dev)
print("✅ 16.2 passed")

### Running on Google Colab (free T4 GPU) or Kaggle
1. Push `medsearch-lab` to GitHub (private is fine; Colab can authenticate with a token, or temporarily make the repo public).
2. In Colab, go to **Runtime → Change runtime type → T4 GPU**. Then:
```python
!git clone https://github.com/<you>/medsearch-lab.git
%cd medsearch-lab
!pip install -q -r requirements.txt medmnist
from google.colab import drive; drive.mount("/content/drive")      # save checkpoints here: Colab sessions die
```
3. Train with `device="cuda"`. **Checkpoint every epoch** with `torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": e}, path)`, so a disconnect doesn't cost you the run.
4. *(Optional speed-up)* Mixed precision: wrap the forward pass in `with torch.autocast("cuda", dtype=torch.float16):` and use `torch.amp.GradScaler()` for the backward pass.
5. **Kaggle Notebooks** have a weekly free GPU quota and don't disconnect as often. They're a good alternative.

**Fine-tune on the GPU:** `ResNetClassifier(pretrained=True, freeze_backbone=False)`, lr `1e-4`, 5–10 epochs, 3 seeds. Write the numbers into the table on Day 3. *(If you have no GPU access this week, skip it and note why. Frozen + SmallCNN is enough for the milestone.)*

---
# Day 3: Variance across seeds (1.5 h)

The same code with a different random seed gives a different result (initialisation, data order, dropout). **A single run is an anecdote.** Report the mean ± std over at least 3 seeds. If two models' ranges overlap heavily, you haven't shown a difference.

### Exercise 16.3: `run_seeds`
Write `run_seeds(make_model, seeds, epochs, lr)`. For each seed: `set_seed(seed)`, build `make_model()`, `fit` it (on the device from `get_device()`), and compute the validation AUROC. Return the list of AUROCs. Then fill the table for SmallCNN and the frozen ResNet.

For the frozen model, pass only the trainable parameters to the optimiser (your `fit` may need a `params=` argument, or filter with `p.requires_grad`).

In [ ]:
def run_seeds(make_model, seeds=(0, 1, 2), epochs=EPOCHS, lr=1e-3):
    raise NotImplementedError

In [ ]:
import pandas as pd
results = {
    "SmallCNN": run_seeds(lambda: imodels.SmallCNN()),
    "ResNet18 frozen": run_seeds(lambda: imodels.ResNetClassifier(pretrained=PRETRAINED, freeze_backbone=True), lr=1e-3),
}
table = pd.DataFrame({k: {"val AUROC mean": np.mean(v), "std": np.std(v, ddof=1), "runs": len(v)} for k, v in results.items()}).T
print(table.round(4))
assert all(len(v) == 3 for v in results.values())
table.to_csv(CAP / "results" / "week16_seeds_val.csv")
print("✅ 16.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Inside the loop: `itrain.set_seed(seed); m = make_model(); m, _ = itrain.fit(m, ds['train'], val_X, val_y, epochs=epochs, lr=lr, seed=seed, device=dev)`, then `roc_auc_score(val_y, itrain.predict_proba(m, val_X, device=dev))`.

</details>


✍️ *Is the frozen ImageNet ResNet better than a tiny CNN trained from scratch? Why might ImageNet features transfer only partly to 28×28 X-rays? What would you expect from fine-tuning?* (double-click to edit)



---
# Day 4: Calibration (2 h)

A model is **calibrated** if, among the cases where it says "80% pneumonia", about 80% really have pneumonia. AUROC ignores calibration completely (it only looks at the ranking), but a clinician reading a probability cares a lot. Modern networks are often **over-confident** (Guo et al., 2017), and class-weighted training shifts probabilities as well.

**Expected calibration error (ECE):** split the predictions into `n_bins` equal-width confidence bins on [0, 1]; in each bin, compare the mean predicted probability of class 1 with the observed fraction of class 1; ECE is the bin-size-weighted mean of the absolute gaps.
$$\text{ECE} = \sum_b \frac{|B_b|}{n}\,\Big|\,\bar y_b - \bar p_b\,\Big|$$

### Exercise 16.4: `capstone/medlab/imaging/calibration.py`
```python
expected_calibration_error(y, p, n_bins=10) -> float      # bins [0, 0.1), [0.1, 0.2), ..., [0.9, 1.0]; empty bins are skipped
reliability_curve(y, p, n_bins=10) -> (bin_mean_pred, bin_frac_pos, bin_counts)   # for plotting
```
Add `tests/test_calibration.py`: a perfectly calibrated synthetic example (y ~ Bernoulli(p)) should have ECE ≈ 0, and always predicting 1.0 when half are positive should have ECE = 0.5.

In [ ]:
# Implement calibration.py and its tests, then run the checker.

In [ ]:
calib = fresh("medlab.imaging.calibration")
assert run_tests("tests/test_calibration.py") == 0
rng = np.random.default_rng(0)
p = rng.random(20000); y = (rng.random(20000) < p).astype(int)
assert calib.expected_calibration_error(y, p) < 0.02
assert abs(calib.expected_calibration_error(np.array([0, 1] * 50), np.ones(100)) - 0.5) < 1e-12

itrain.set_seed(0)
best, _ = itrain.fit(imodels.SmallCNN(), ds["train"], val_X, val_y, epochs=EPOCHS, seed=0, device=dev)
pv = itrain.predict_proba(best, val_X, device=dev)
ece = calib.expected_calibration_error(val_y, pv)
mp, fp, cnt = calib.reliability_curve(val_y, pv)
plt.plot([0, 1], [0, 1], "k--"); plt.plot(mp, fp, "o-"); plt.xlabel("mean predicted P(pneumonia)"); plt.ylabel("observed fraction")
plt.title(f"Reliability diagram (val), ECE = {ece:.3f}"); plt.show()
print("✅ 16.4 passed (milestone M6)")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)` puts p = 1.0 into the last bin.

</details>

<details><summary>Hint 2</summary>

For each bin b with `mask = bins == b` and `mask.any()`: `gap = abs(y[mask].mean() - p[mask].mean())`, weighted by `mask.mean()`.

</details>


✍️ *Is the model over- or under-confident? Class weighting pushes predictions towards the minority class: how does that show up in the diagram? *(Optional: implement temperature scaling, fitting one scalar T on the validation logits, and compare ECE before and after.)** (double-click to edit)



---
# Month 4 reflection
1. Update the tracker (Neural networks, Vision).
2. Write the Month 4 summary in `LOG.md`.
3. Commit: `Add ResNet transfer-learning model`, `Add calibration metrics`, `Add multi-seed results`.
4. **Send me** the repo link.

**Spaced recall:** (W7) MAP ↔ L2. (W10) the BM25 formula. (W12) Holm correction. (W13) why ReLU. (W14) the 6 training bugs. (W15) the receptive field. And this week: frozen vs fine-tuned; why multiple seeds; what ECE measures that AUROC doesn't.

✍️ *Your answer:* (double-click to edit)

